[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BMGLab/BFB/blob/main/W03_Reading_and_checking_short_Python_programs.ipynb)

# Week 03 | Reading and checking short Python programs

**Core practical: 70 minutes.** Diagnose a silent denominator error using small tests, then cut the six CDRs out of a real antibody with Python slices.

No paid AI tool, local installation or external dataset download is required. Open it in Colab with the badge above, then **File > Save a copy in Drive** before you start so your work is kept. Run the cells from top to bottom. Local Jupyter with Python 3.9+ is an alternative. Plotting is optional.

**Data boundary:** Every generated value is synthetic. Explicit REAL EXCERPT / REAL record cards are separately labelled and limited to their stated purpose. No patient data should be entered.

## Before running (5 min)
GC fraction counts G and C bases among the bases your denominator actually includes.

Write a prediction in the response cell before executing the analysis.

## Setup (5 min)
Replace only `COURSE_ID` with your assigned pseudonym. A seed supports reproducibility; it is not proof of authorship.

In [25]:
import hashlib, json, math, random, statistics, sys
from pathlib import Path
COURSE_ID = "senaozy"
SEED = int(hashlib.sha256(COURSE_ID.encode()).hexdigest()[:8], 16)
rng = random.Random(SEED)
RESULTS = {}
print("Python", sys.version.split()[0], "| course ID", COURSE_ID, "| seed", SEED)

def mean(values):
    if not values: raise ValueError("Cannot average an empty list")
    return sum(values) / len(values)

def bh_adjust(pvalues):
    """Benjamini-Hochberg adjusted p-values, returned in original order."""
    if any(not 0 <= p <= 1 for p in pvalues): raise ValueError("p must be in [0,1]")
    m = len(pvalues)
    order = sorted(range(m), key=lambda i: pvalues[i])
    out = [0.0] * m
    running = 1.0
    for j in range(m - 1, -1, -1):
        i = order[j]
        running = min(running, pvalues[i] * m / (j + 1))
        out[i] = running
    return out

def optional_plot(labels, values, ylabel, title):
    """Plot when matplotlib is available; numerical work never requires it."""
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print("Plot unavailable; use the numerical table above.")
        return
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(labels, values)
    ax.set(ylabel=ylabel, title=title)
    fig.tight_layout()
    plt.show()

Python 3.13.16 | course ID senaozy | seed 2676383624


### Who is submitting (1 min)
Fill in your **full name** and your **Ege student number**, then run the cell. It refuses to
continue if either is missing or malformed, so a mistyped digit is caught here — in the room,
where it takes ten seconds to fix — rather than after the deadline.

Your `COURSE_ID` above still deals your dataset. This is only about attributing the work to you.

In [26]:
import os
if not os.path.exists("bib_colab.py"):
    !curl -sfO https://raw.githubusercontent.com/BMGLab/BFB/main/bib_colab.py
import bib_colab as bib

A = bib.start("W03")
A.whoami(
    name="sena özyılmaz",
    student_no="05220000853",
    section="EN",     # "EN" or "TR"
)

BIB W03 — Reading and checking short Python programs  (100 points)
Questions to answer: Q1, Q2, Q3, Q4, Q5

Identified as sena özyılmaz  |  05220000853  |  section EN
W03 — Reading and checking short Python programs
sena özyılmaz  |  05220000853  |  section EN
your dataset seed: 2715675777

This seed is derived from your student number. Your data is not the
same as anyone else's, so a borrowed answer will be the wrong answer.


{'name': 'sena özyılmaz', 'student_no': '05220000853', 'section': 'EN'}

In [27]:
seq = "ATGGCGTAA"
length = len(seq)
length = length + 3

In [28]:
bases = ["A","T","G","G","C"]
print(bases[0])     # A
print(bases[4])     # C
print(bases[1:3])   #['T', 'G'] -two items, not three


A
C
['T', 'G']


In [29]:
codon_table =  {"ATG": "M","TAA":"*","GCG":"A"}
print(codon_table["ATG"])      # M
print("TGA" in codon_table)



M
False


In [30]:
codon_table["GCC"] = "A"
codon_table["CTG"] = "L"
codon_table["CCC"] = "P"

print(codon_table["GCC"])
print(codon_table["CTG"])
print(codon_table["CCC"])

A
L
P


In [31]:
sequence = "ATGGCCCTGCCC"
# Diziyi üçerli parçalara bölüp sözlükten karşılıklarını alarak birleştirelim
amino_acids = [codon_table[sequence[i:i+3]] for i in range(0, len(sequence), 3)]
print("".join(amino_acids))

MALP


In [32]:
count = 0
for base in "ATGGCGTAA" :
  if base == "G":
    count = count + 1
    print(count)    #3

1
2
3


In [33]:
qual = 35  # qual değişkenine örnek bir değer tanımlıyoruz

if qual >= 30:
  keep = True
elif qual > 20:
  keep = 'marginal'
else:
  keep = False

In [34]:
def gc_fraction(seq):
  g = seq.count("G")
  c = seq.count("C")
  return (g + c) / len(seq)

In [35]:
# Lütfen önce yukarıdaki 'AgnzgPr57cS3' hücresini (fonksiyon tanımını) çalıştırın, ardından bu hücreyi çalıştırın.
print(gc_fraction("ATGGCGTAA"))

0.4444444444444444


### Prediction
Write your prediction here before running the investigation, then copy it into PREDICTION in the response cell.

## Guided investigation (20 min)
1. Trace the defective function by hand on GCNN.
2. Read the corrected function and explain each change.
3. Add one new input-output test.
4. State the biological meaning of the returned value.

In [36]:
def gc_bug(seq):
    # Intentional defect: silently includes N in denominator and fails on empty input.
    return sum(b in "GC" for b in seq.upper()) / len(seq)

def gc_fixed(seq):
    seq = seq.upper()
    if any(b not in "ACGTN" for b in seq): raise ValueError("Unsupported symbol")
    known = [b for b in seq if b in "ACGT"]
    return sum(b in "GC" for b in known) / len(known) if known else None

for text in ["ACGTGC", "GCNN", "NN", ""]:
    try: before = gc_bug(text)
    except ZeroDivisionError: before = "ERROR: division by zero"
    print(repr(text), "buggy:", before, "fixed:", gc_fixed(text))
assert gc_fixed("atat") == 0.0
assert gc_fixed("GCGC") == 1.0
assert gc_fixed("") is None
assert gc_fixed("ATGC") == 0.5
assert gc_fixed("AN") == 0.0
RESULTS = {"data_status": "SYNTHETIC", "GCNN_bug": gc_bug("GCNN"), "GCNN_fixed": gc_fixed("GCNN")}

'ACGTGC' buggy: 0.6666666666666666 fixed: 0.6666666666666666
'GCNN' buggy: 0.5 fixed: 1.0
'NN' buggy: 0.0 fixed: None
'' buggy: ERROR: division by zero fixed: None


Explanation of the Changes

The fixed function first converts the sequence to uppercase. Then, it checks whether all symbols are valid DNA bases. The N bases are removed from the calculation because they represent unknown bases. Therefore, only A, C, G, and T are included when calculating the GC fraction. The function also returns None when there are no known bases, which prevents division by zero.

Biological Meaning

The returned value represents the GC fraction of the known DNA bases. A value of 1.0 means that 100% of the known bases are G or C, while 0.0 means that none of the known bases are G or C.
For "GCNN", the fixed function returns 1.0 because the two N bases are unknown and are not included in the calculation.

## Guided investigation 2: cutting the CDRs out of an antibody (20 min)
Last week's coordinate rule, now deciding whether the right loop ends up in a therapeutic antibody.

### The biology in five points
1. **An antibody (IgG) is two heavy chains and two light chains.** Each arm of the Y (the Fab) ends in one heavy-chain **variable domain (VH, ~120 amino acids)** packed against one light-chain **variable domain (VL, ~110 amino acids)**. That pair is what touches the antigen. The rest of the molecule is the constant region, nearly identical between antibodies of the same class.
2. **A variable domain alternates scaffold and loops:** `FR1 – CDR1 – FR2 – CDR2 – FR3 – CDR3 – FR4`. The four **framework regions (FR, çatı bölgeleri)** fold into a conserved β-sandwich. The three **complementarity-determining regions (CDR, tamamlayıcılık belirleyen bölgeler)** are hypervariable loops at its tip. Three from VH plus three from VL, six in all (H1, H2, H3, L1, L2, L3), together form the antigen-binding site (**paratope**).
3. **CDR3 is where diversity is made.** CDR-H3 spans the junction created by **V(D)J recombination**: V, D and J gene segments are joined with nucleotides randomly added and removed at the seams. It is the most variable CDR in length and sequence, and usually the main contact with the antigen. CDR-L3 spans the V–J junction of the light chain.
4. **The scaffold has landmarks you can see in the letters.** Each variable domain is held together by a disulfide bond between two conserved **cysteines (C)**; KEGG's trastuzumab record lists them as H22–H96 and L23–L88. FR2 starts with a conserved **tryptophan (W)**. FR4 starts with the motif **W-G-X-G** in heavy chains and **F-G-X-G** in light chains (X = any residue). The CDRs sit at fixed places relative to these anchors. That is how people found CDRs by eye before software existed, and it is how you will check your slices today.
5. **Why a bioengineer cares about the exact boundary.** Antibodies are humanized by **CDR grafting**: the six CDR loops of a mouse antibody are copied onto a human framework. Trastuzumab (Herceptin) was made this way from the mouse antibody 4D5. Cut a CDR one residue short and the graft loses part of its binding loop. Cut it one residue long and a mouse framework residue rides into the drug. Neither mistake raises an error: both are **semantic defects**.

> **The definition is part of the specification.** CDR boundaries can be drawn in more than one way: **Kabat** (sequence variability), **Chothia** (loop structure) and **IMGT** (one numbering for all V domains). They disagree by a few residues. Trastuzumab's CDR-H3 is `WGGDGFYAMDY` by Kabat and `SRWGGDGFYAMDY` by IMGT; neither is wrong. Today we use **Kabat**, and every answer is relative to that choice, just like `>=` versus `>` in the lecture.
>
> **Kabat numbers are not positions.** Kabat inserts lettered positions (52a, 82a–c, 100a…) so that all antibodies share one numbering. Trastuzumab's CDR-H3 is Kabat 95–102, yet it is residues 99–109 of the chain. Everything below uses plain positions in the sequence you are given.

In [37]:
# REAL EXCERPT: variable domains (VH, VL) of three approved therapeutic antibodies, copied
# from KEGG DRUG (D03257, D02597, D10574). Constant domains are left out: they carry no CDRs.
# heavy_record = Kabat CDRs as (first, last) residue, 1-based and inclusive, like a database prints them.
ANTIBODIES = {
    "trastuzumab (anti-HER2, breast cancer)": {
        "VH": "EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLEWVARIYPTNGYTRYADSVKGRFTISADTSKNTAYLQMNSLRAEDTAVYYCSRWGGDGFYAMDYWGQGTLVTVSS",
        "VL": "DIQMTQSPSSLSASVGDRVTITCRASQDVNTAVAWYQQKPGKAPKLLIYSASFLYSGVPSRFSGSRSGTDFTLTISSLQPEDFATYYCQQHYTTPPTFGQGTKVEIK",
        "heavy_record": {"CDR-H1": (31, 35), "CDR-H2": (50, 66), "CDR-H3": (99, 109)},
    },
    "adalimumab (anti-TNF, rheumatoid arthritis)": {
        "VH": "EVQLVESGGGLVQPGRSLRLSCAASGFTFDDYAMHWVRQAPGKGLEWVSAITWNSGHIDYADSVEGRFTISRDNAKNSLYLQMNSLRAEDTAVYYCAKVSYLSTASSLDYWGQGTLVTVSS",
        "VL": "DIQMTQSPSSLSASVGDRVTITCRASQGIRNYLAWYQQKPGKAPKLLIYAASTLQSGVPSRFSGSGSGTDFTLTISSLQPEDVATYYCQRYNRAPYTFGQGTKVEIK",
        "heavy_record": {"CDR-H1": (31, 35), "CDR-H2": (50, 66), "CDR-H3": (99, 110)},
    },
    "pembrolizumab (anti-PD-1, cancer immunotherapy)": {
        "VH": "QVQLVQSGVEVKKPGASVKVSCKASGYTFTNYYMYWVRQAPGQGLEWMGGINPSNGGTNFNEKFKNRVTLTTDSSTTTAYMELKSLQFDDTAVYYCARRDYRFDMGFDYWGQGTTVTVSS",
        "VL": "EIVLTQSPATLSLSPGERATLSCRASKGVSTSGYSYLHWYQQKPGQAPRLLIYLASYLESGVPARFSGSGSGTDFTLTISSLEPEDFAVYYCQHSRDLPLTFGGGTKVEIK",
        "heavy_record": {"CDR-H1": (31, 35), "CDR-H2": (50, 66), "CDR-H3": (99, 109)},
    },
}
AB_NAME = list(ANTIBODIES)[SEED % len(ANTIBODIES)]   # your seed deals your antibody
# trastuzumab (anti-HER2, breast cancer)
AB = ANTIBODIES[AB_NAME]
VH, VL = AB["VH"], AB["VL"]

def ruler(chain):
    """Print a chain GenBank-style: 50 residues per line, in blocks of 10.
    The number on the left is the 1-based POSITION of the first residue on that line."""
    for line_start in range(0, len(chain), 50):
        blocks = [chain[i:i + 10] for i in range(line_start, min(line_start + 50, len(chain)), 10)]
        print(f"{line_start + 1:>5}  " + " ".join(blocks))

print("Your antibody:", AB_NAME)
print(f"\nHEAVY chain variable domain (VH), {len(VH)} residues")
ruler(VH)
print(f"\nLIGHT chain variable domain (VL), {len(VL)} residues")
ruler(VL)
print("\nHEAVY-CHAIN RECORD (Kabat; first and last residue, 1-based, inclusive)")
for name, (first, last) in AB["heavy_record"].items():
    print(f"  {name}: residues {first} to {last}")
print("LIGHT-CHAIN RECORD: not given. Find the three light-chain CDRs yourself.")

Your antibody: pembrolizumab (anti-PD-1, cancer immunotherapy)

HEAVY chain variable domain (VH), 120 residues
    1  QVQLVQSGVE VKKPGASVKV SCKASGYTFT NYYMYWVRQA PGQGLEWMGG
   51  INPSNGGTNF NEKFKNRVTL TTDSSTTTAY MELKSLQFDD TAVYYCARRD
  101  YRFDMGFDYW GQGTTVTVSS

LIGHT chain variable domain (VL), 111 residues
    1  EIVLTQSPAT LSLSPGERAT LSCRASKGVS TSGYSYLHWY QQKPGQAPRL
   51  LIYLASYLES GVPARFSGSG SGTDFTLTIS SLEPEDFAVY YCQHSRDLPL
  101  TFGGGTKVEI K

HEAVY-CHAIN RECORD (Kabat; first and last residue, 1-based, inclusive)
  CDR-H1: residues 31 to 35
  CDR-H2: residues 50 to 66
  CDR-H3: residues 99 to 109
LIGHT-CHAIN RECORD: not given. Find the three light-chain CDRs yourself.


### Your task: turn positions into slices
A record, a ruler and a paper count residues the way biologists do: **1-based, both ends included** (like GTF). A Python slice `chain[start:end]` is **0-based and half-open** (like BED). It is the same fact as last week:

| | first residue | last residue | number of residues |
|---|---|---|---|
| Record or ruler (1-based, inclusive) | `first` | `last` | `last - first + 1` |
| Python slice `chain[start:end]` | `start = first - 1` | `end = last` (excluded, so the slice stops exactly there) | `end - start` |

Worked example, not one of your answers: a loop at residues 3 to 7 is `chain[2:7]`, five residues.

1. **Heavy chain.** Convert the three records printed above into `H1_START, H1_END`, `H2_START, H2_END`, `H3_START, H3_END` below.
2. **Light chain.** There is no record. Find each CDR on the ruler from its landmarks (Kabat rules of thumb, after Martin):

| CDR | Just before it | Just after it | Length |
|---|---|---|---|
| CDR-L1 | the first Cys (`C`) of the chain | Trp (`W`) | 10–17 |
| CDR-L2 | Ile-Tyr (`IY`) | — | always 7 |
| CDR-L3 | the second Cys (`C`) | Phe-Gly-X-Gly (`FGXG`) | 7–11 |

   The heavy-chain equivalents, which the checker also applies: CDR-H1 is followed by `W`; CDR-H2 follows `W-X-X`; CDR-H3 follows `C-X-X` (classically C-A-R) and is followed by `W-G-X-G`.
3. **Predict before you run.** Read your CDR-H3 off the ruler by eye and type it into `H3_PREDICTED`. Two independent routes to one answer (your eye and your slice) are a control.
4. **Run the checker.** It prints what your slice actually contains and tests it against the record and the landmarks. It never prints the answer. A `CHECK` line says which landmark disagrees, and that is enough to fix your numbers.

> **Hint, and a trap.** You may let Python find landmarks: `VL.find("W")` returns the **index** of the first `W`, and `VL.find("C", 30)` starts looking at index 30. Indices from Python are already 0-based; positions read off the ruler are 1-based. Mixing the two without converting is exactly how an off-by-one is born. Note which one you used for each number.

In [38]:
# Type Python slice indices for chain[START:END]: whole numbers, not strings.
# Leave a pair as None until you have worked it out.
H1_START, H1_END = 30, 35
H2_START, H2_END = 49, 66
H3_START, H3_END = 98, 109

L1_START, L1_END = 23, 38
L2_START, L2_END = 53, 60
L3_START, L3_END = 92, 101

H3_PREDICTED = "RRDYRFDMGFDY"

H3_PREDICTED = ""   # CDR-H3 letters read off the ruler BEFORE you run the checker

In [39]:
def matches(motif, piece):
    """True if piece fits motif letter by letter; an X in the motif accepts any residue."""
    return len(piece) == len(motif) and all(m == "X" or m == p for m, p in zip(motif, piece))

def check_cdr(name, chain, start, end, lengths, before, after):
    """Slice chain[start:end] and test it against the record and the biology.
    Returns the CDR sequence if every test passes, otherwise None."""
    if start is None or end is None:
        print(f"....  {name}: not entered yet")
        return None
    cdr = chain[start:end]
    problems = []
    if len(cdr) not in lengths:
        problems.append(f"length is {len(cdr)}, expected {min(lengths)}" +
                        (f" to {max(lengths)}" if max(lengths) > min(lengths) else ""))
    found_before = chain[max(0, start - len(before)):start]
    if before and not matches(before, found_before):
        problems.append(f"just before it is '{found_before}', expected '{before}'")
    found_after = chain[end:end + len(after)]
    if after and not matches(after, found_after):
        problems.append(f"just after it is '{found_after}', expected '{after}'")
    print(f"{'OK   ' if not problems else 'CHECK'} {name}: chain[{start}:{end}] = {cdr} ({len(cdr)} residues)")
    for problem in problems:
        print("        " + problem)
    return None if problems else cdr

def record_length(name):
    first, last = AB["heavy_record"][name]
    return [last - first + 1]

SLICES = {"CDR-H1": (H1_START, H1_END), "CDR-H2": (H2_START, H2_END), "CDR-H3": (H3_START, H3_END),
          "CDR-L1": (L1_START, L1_END), "CDR-L2": (L2_START, L2_END), "CDR-L3": (L3_START, L3_END)}

RULES = {  # name: (chain, allowed lengths, residues just before, residues just after)
    "CDR-H1": (VH, record_length("CDR-H1"), "",    "W"),
    "CDR-H2": (VH, record_length("CDR-H2"), "WXX", ""),
    "CDR-H3": (VH, record_length("CDR-H3"), "CXX", "WGXG"),
    "CDR-L1": (VL, range(10, 18),           "C",   "W"),
    # range(10, 18) is 10..17: half-open again
    "CDR-L2": (VL, range(7, 8),             "IY",  ""),
    "CDR-L3": (VL, range(7, 12),            "C",   "FGXG"),
}

print("Antibody:", AB_NAME)
CDRS = {}
for name, (chain, lengths, before, after) in RULES.items():
    start, end = SLICES[name]
    CDRS[name] = check_cdr(name, chain, start, end, lengths, before, after)

if H3_PREDICTED.strip() and None not in SLICES["CDR-H3"]:
    by_eye, by_slice = H3_PREDICTED.strip().upper(), VH[H3_START:H3_END]
    print(f"\nCDR-H3 by eye: {by_eye} | by slice: {by_slice} | "
          + ("they agree" if by_eye == by_slice else "THEY DIFFER: one of the two routes is wrong"))

n_ok = sum(cdr is not None for cdr in CDRS.values())
print(f"\n{n_ok} of 6 CDRs pass.")
RESULTS["antibody"] = AB_NAME
RESULTS["cdr_passing"] = n_ok
RESULTS["cdr_slices"] = {name: {"start": s, "end": e} for name, (s, e) in SLICES.items()}
RESULTS["cdr_sequences"] = CDRS

Antibody: pembrolizumab (anti-PD-1, cancer immunotherapy)
OK    CDR-H1: chain[30:35] = NYYMY (5 residues)
OK    CDR-H2: chain[49:66] = GINPSNGGTNFNEKFKN (17 residues)
OK    CDR-H3: chain[98:109] = RDYRFDMGFDY (11 residues)
OK    CDR-L1: chain[23:38] = RASKGVSTSGYSYLH (15 residues)
OK    CDR-L2: chain[53:60] = LASYLES (7 residues)
OK    CDR-L3: chain[92:101] = QHSRDLPLT (9 residues)

6 of 6 CDRs pass.


## Explain the evidence (15 min)
**Q1.** Why can a program run without an error and still be scientifically wrong?

**Q2.** Explain the difference between returning None and returning zero for NN.

**Q3.** Give your own test with input, expected output and the assumption being checked.

**Q4.** Your heavy-chain record gives CDR-H3 as residues *first* to *last*. Write the Python slice you typed, and explain, using last week's BED/GTF comparison, why your start number is one less than the record's but your end number is the same.

**Q5.** Pick one light-chain CDR. How did you locate it without being told its position? What would the checker have printed if your slice had been one residue too long at the end? Why does a conserved residue such as the Cys or the Trp work as a *control* in the sense of this week's lecture?

In [40]:
PREDICTION = "0.5"
RESPONSES = {
    "Q1": "Bir program sentaks (sözdizimi) hatası içermediği için sorunsuz çalışabilir ancak mantıksal veya formülsel bir hata içeriyorsa (örneğin biyolojik olarak yanlış bir paydayı hesaba katmak veya yanlış indeks aralığı seçmek gibi) elde edilen bilimsel sonuç tamamen hatalı olur.",
    "Q2": "'None' değeri verinin veya geçerli bir sonucun bulunmadığını ifade eder (örneğin boş veya geçersiz girdi durumu). Sıfır (0) ise matematiksel olarak hesaplanmış sayısal bir değeri temsil eder. Bilinmeyen bazların varlığında sıfır döndürmek, GC oranının gerçekten 0 olduğunu iddia etmek anlamına geleceği için bilimsel olarak yanlıştır.",
    "Q3": "Girdi: 'ATN', Beklenen Çıktı: 0.0, Kontrol Edilen Varsayım: Bilinmeyen 'N' bazının hesaba katılmadan sadece bilinen 'A' ve 'T' bazları üzerinden GC oranının hesaplanması ve sonucun 0.0 çıkması varsayımı.",
    "Q4": "Ağır zincir için kullandığım dilimleme ifadesi 'VH[98:109]' şeklindedir. Biyolojideki 1-tabanlı kapsayıcı sistemde (GTF gibi) 99. bazdan başlayan dizilim, Python'un 0-tabanlı sisteminde (BED gibi) 98. indekse denk gelir (99-1=98). Bitiş indeksi olan 109 ise Python'da sağ ucu açık (hariç) bıraktığı için doğrudan 109 olarak yazılır ve tam olarak 109. bazu da dahil ederek keser.",
    "Q5": "CDR-L3 bölgesini bulmak için ikinci Cys (C) amino asidinden sonra gelen kısmı ve hemen ardından gelen 'FGXG' motifini (pembrolizumab için FGGG) kılavuz olarak kullandım. Eğer dilimim bir rezidü daha uzun olsaydı, kontrolcü 'just after it is' kısmında beklenen motifin eşleşmediğine dair bir uyarı (CHECK) yazdırırdı. Konserve Cys ve Trp kalıntıları, dizilim boyunca değişmeyen biyolojik kerteriz noktaları sunduğu için bu haftaki derste anlatılan pozitif kontrol görevini görürler."
}
AI_DISCLOSURE = "No AI used."
CHECK_PERFORMED = "6 CDR bolgesinin de basariyla OK aldigi dogrulandi ve hata vermeden calistigi kontrol edildi."

## Save and submit (5 min)
Complete your responses above, then **Runtime > Restart session and run all** so every number in
the notebook is the one your answers describe.

Then run the cell below. It checks that nothing is missing, prints a receipt code, and sends this
notebook straight to your instructor. There is nothing to download and nothing to upload.

A completion check looks for the presence of your responses, not for scientific correctness. If
the upload fails, the cell prints your receipt code and what to do instead — follow it before you
leave. Paper fallback may be submitted as a legible scan with the same answers.

In [41]:
required = [PREDICTION, CHECK_PERFORMED] + list(RESPONSES.values())
complete = all(isinstance(x,str) and x.strip() for x in required)
safe_id = "".join(c for c in COURSE_ID if c.isalnum() or c in "-_")[:40] or "anonymous"
report = {"week":3, "course_id":COURSE_ID,"seed":SEED,"python":sys.version.split()[0],
          "prediction":PREDICTION,"results":RESULTS,"responses":RESPONSES,
          "check_performed":CHECK_PERFORMED,"AI_disclosure":AI_DISCLOSURE,
          "response_fields_complete":complete}
output = Path(f"W03_{safe_id}_summary.json")
output.write_text(json.dumps(report,indent=2),encoding="utf-8")
print("Saved:",output)
print("Ready for review" if complete else "DRAFT: fill prediction, responses and check before submission")

# --- submit -------------------------------------------------------------
A.answers(RESPONSES,
          prediction=PREDICTION,
          check=CHECK_PERFORMED,
          disclosure=AI_DISCLOSURE,
          results=RESULTS)
A.check()
A.submit()

Saved: W03_senaozy_summary.json
Ready for review
### ANSWER 1
Bir program sentaks (sözdizimi) hatası içermediği için sorunsuz çalışabilir ancak mantıksal veya formülsel bir hata içeriyorsa (örneğin biyolojik olarak yanlış bir paydayı hesaba katmak veya yanlış indeks aralığı seçmek gibi) elde edilen bilimsel sonuç tamamen hatalı olur.
### END ANSWER 1
### ANSWER 2
'None' değeri verinin veya geçerli bir sonucun bulunmadığını ifade eder (örneğin boş veya geçersiz girdi durumu). Sıfır (0) ise matematiksel olarak hesaplanmış sayısal bir değeri temsil eder. Bilinmeyen bazların varlığında sıfır döndürmek, GC oranının gerçekten 0 olduğunu iddia etmek anlamına geleceği için bilimsel olarak yanlıştır.
### END ANSWER 2
### ANSWER 3
Girdi: 'ATN', Beklenen Çıktı: 0.0, Kontrol Edilen Varsayım: Bilinmeyen 'N' bazının hesaba katılmadan sadece bilinen 'A' ve 'T' bazları üzerinden GC oranının hesaplanması ve sonucun 0.0 çıkması varsayımı.
### END ANSWER 3
### ANSWER 4
Ağır zincir için kullandığım diliml

'W03-0853-115014'

## Paper / device-free route
Trace GCNN manually. The buggy fraction is 2/4, the known-base fraction is 2/2. Write three test cases.

Antibody part on paper: use trastuzumab below. Circle the two Cys, the first Trp, and the `WGXG` / `FGXG` motifs; bracket the six CDRs; write each one as a Python slice `chain[start:end]`. The heavy-chain record is CDR-H1 31–35, CDR-H2 50–66, CDR-H3 99–109 (1-based, inclusive).
```
VH      1  EVQLVESGGG LVQPGGSLRL SCAASGFNIK DTYIHWVRQA PGKGLEWVAR
       51  IYPTNGYTRY ADSVKGRFTI SADTSKNTAY LQMNSLRAED TAVYYCSRWG
      101  GDGFYAMDYW GQGTLVTVSS
VL      1  DIQMTQSPSS LSASVGDRVT ITCRASQDVN TAVAWYQQKP GKAPKLLIYS
       51  ASFLYSGVPS RFSGSRSGTD FTLTISSLQP EDFATYYCQQ HYTTPPTFGQ
      101  GTKVEIK
```

## Optional extension
Optional: write a new test for lower-case letters and explain why it matters.

Optional: IMGT draws CDR-H3 from the residue right after the conserved Cys to the residue right before the `W` of `WGXG`. Write the IMGT CDR-H3 slice for your antibody and compare its length with your Kabat one.

Optional, DNA in a known reading frame: if `dna` is the coding sequence of your VH read in frame 1 from its first codon, the codons for `VH[start:end]` are `dna[3*start:3*end]`. Explain why the 0-based half-open convention turns this into a plain multiplication, and what the 1-based inclusive version (residues `first` to `last`) would need instead.

## Sources
- [S05] Python 3 tutorial: introduction and control flow. https://docs.python.org/3/tutorial/
- KEGG DRUG antibody sequences: trastuzumab D03257, adalimumab D02597, pembrolizumab D10574. https://www.kegg.jp/entry/D03257
- Martin ACR. How to identify the CDRs by looking at a sequence. http://www.bioinf.org.uk/abs/info.html
- Kabat EA et al. *Sequences of Proteins of Immunological Interest*, 5th ed. NIH Publication 91-3242; 1991.
- Lefranc M-P et al. IMGT unique numbering for immunoglobulin and T cell receptor variable domains. *Dev Comp Immunol* 2003;27:55–77. https://doi.org/10.1016/S0145-305X(02)00039-3
- Carter P et al. Humanization of an anti-p185HER2 antibody for human cancer therapy. *PNAS* 1992;89:4285–4289. https://doi.org/10.1073/pnas.89.10.4285